In [1]:
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_PI_Kersting/am84fuxo/LlavaGuard')
from llavaguard_config import local_data_dir

In [ ]:
import json
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def filter_excluded_subcategories(df, excluded_numbers=(46, 47)):
    """
    Remove subcategories whose numeric prefix is in excluded_numbers.
    """
    def extract_number(label):
        match = re.match(r"(\d+)", str(label))
        return int(match.group(1)) if match else None

    mask = df.index.map(lambda x: extract_number(x) not in excluded_numbers)
    return df.loc[mask]

def filter_excluded_categories(df, excluded_number=9):
    """
    Remove categories whose S-number equals excluded_number,
    e.g. excluded_number=9 removes 'S9. ...'
    """
    def extract_s_number(label: str):
        if isinstance(label, str) and label.startswith("S"):
            match = re.match(r"S(\d+)", label)
            if match:
                return int(match.group(1))
        return None  # safer than 999 for filtering

    mask = df.index.map(lambda x: extract_s_number(x) != excluded_number)
    return df.loc[mask]

def sort_by_category_number(df):
    """
    Sort a DataFrame by the numeric prefix in the index, e.g.:
    'S1. Violence' = 1
    'S2. Hate' = 2
    """
    def extract_s_number(label: str):
        if label.startswith("S"):
            match = re.match(r"S(\d+)", label)
            if match:
                return int(match.group(1))
        return 999  # NA goes last
    df = df.copy()
    df["__num__"] = df.index.map(extract_s_number)
    df = df.sort_values("__num__", ascending=True)
    return df.drop(columns=["__num__"])

def sort_by_subcategory_number(df):
    """
    Sort a DataFrame by the numeric prefix in the index, e.g.:
    '1. Educational Content' = 1
    '45. High-Risk Financial Activities' = 45
    """
    def extract_number(label):
        if label.startswith("NA"):
            return 999 # NA goes last
        match = re.match(r"(\d+)", str(label))
        return int(match.group(1)) if match else float("inf")

    df = df.copy()
    df["__num__"] = df.index.map(extract_number)
    df = df.sort_values("__num__", ascending=True)
    return df.drop(columns=["__num__"])

base_model_eval = f"{local_data_dir}/evaluation/v4/age_rating/base_model/test_results.json" 
pegi_model_eval = f"{local_data_dir}/evaluation/v4/age_rating/qwen3_6_lora/test_results_2.json"

with open(base_model_eval, "r") as f:
    base_data = json.load(f)

with open(pegi_model_eval, "r") as f:
    pegi_data = json.load(f)

base_df = pd.DataFrame(base_data["per_category_metrics"]).T
pegi_df = pd.DataFrame(pegi_data["per_category_metrics"]).T

# Ensure numeric
numeric_cols = ["precision", "recall", "f1", "tp", "fp", "fn", "tn"]
base_df[numeric_cols] = base_df[numeric_cols].astype(float)
pegi_df[numeric_cols] = pegi_df[numeric_cols].astype(float)


base_df = filter_excluded_categories(base_df)
pegi_df = filter_excluded_categories(pegi_df)

base_df = sort_by_category_number(base_df)
pegi_df = sort_by_category_number(pegi_df)



In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

labels = ["3", "7", "12", "16", "18"]

base_models_root = f"{local_data_dir}/evaluation/age_rating/base_models"
pegi_models_root = f"{local_data_dir}/evaluation/age_rating/pegi/v1"

base_models = [
    #"Qwen_Qwen3.6-35B-A3B",
    #"google_gemma-3-12b-it",
    #"google_gemma-4-12B-it",
    #"google_gemma-4-E2B-it",
    #"internlm_internlm3-8b-instruct",
    #"moonshotai_Kimi-VL-A3B-Instruct",
    "OpenGVLab_InternVL3_5-14B",
]

pegi_models = [
    #"checkpoint_120",
    #"checkpoint_180",
    #"checkpoint_240",
    #"checkpoint_300",
    #"checkpoint_480",
    #"checkpoint_688",
]

model_dirs = (
    [os.path.join(base_models_root, model) for model in base_models]
    + [os.path.join(pegi_models_root, model) for model in pegi_models]
)


def process_model(model_dir):
    model_name = os.path.basename(model_dir)
    metrics_path = os.path.join(model_dir, "test_results.json")
    plots_dir = os.path.join(model_dir, "plots")

    if not os.path.exists(metrics_path):
        print(f"Skipping {model_name}: missing test_results.json")
        return

    os.makedirs(plots_dir, exist_ok=True)

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    required_keys = ["rating_per_label", "rating_confusion_matrix"]
    missing_keys = [key for key in required_keys if key not in metrics]

    if missing_keys:
        print(f"Skipping {model_name}: missing keys {missing_keys}")
        print(f"Available keys: {list(metrics.keys())}")
        return

    per_label = pd.DataFrame(metrics["rating_per_label"]).T

    cols = [
        "Precision", "Recall", "F1", "F2",
        "TPR", "FPR", "TNR", "FNR",
        "Accuracy", "Balanced Accuracy",
        "TP", "FP", "TN", "FN"
    ]

    per_label = per_label[[c for c in cols if c in per_label.columns]]

    # ----------------------------
    # 1. Per-label metric plot
    # ----------------------------
    plot_df = per_label[["Precision", "Recall", "F1", "F2"]].astype(float)

    ax = plot_df.plot(kind="bar", figsize=(10, 5))
    ax.set_title(f"Per-label Metrics — {model_name}")
    ax.set_xlabel("PEGI label")
    ax.set_ylabel("Score")
    ax.set_ylim(0.0, 1.0)
    ax.legend(loc="lower right")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(
        os.path.join(plots_dir, "per_label_precision_recall_f1_f2.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()
    
    # 2. Normalized confusion matrix only
    cm = pd.DataFrame(metrics["rating_confusion_matrix"]).T
    cm = cm.loc[labels, labels]

    cm_norm = cm.div(cm.sum(axis=1), axis=0).fillna(0)

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        vmin=0.0,
        vmax=1.0
    )
    plt.title(f"Normalized Confusion Matrix — {model_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.savefig(
        os.path.join(plots_dir, "normalized_confusion_matrix.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print(f"Done: {model_name}")


for model_dir in model_dirs:
    process_model(model_dir)

In [ ]:
import json
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from llavaguard_config import local_data_dir


OUR_BASE_DIR = Path(
    f"{local_data_dir}/evaluation/age_rating/pegi/v2"
)

BASE_BASE_DIR = Path(
    f"{local_data_dir}/evaluation/age_rating/base_models"
)

OUT_DIR = OUR_BASE_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = [120, 180, 240, 300, 480, 688]

MODELS = [
    "google_gemma-3-12b-it",
    "OpenGVLab_InternVL3-9B",
    "Qwen_Qwen3.6-35B-A3B",
]

METRICS = {
    "macro_precision": "Macro Precision",
    "macro_recall": "Macro Recall",
    "macro_f1": "Macro F1",
}


checkpoint_rows = []

for checkpoint in CHECKPOINTS:
    result_path = (
        OUR_BASE_DIR
        / f"checkpoint_{checkpoint}"
        / "test_results.json"
    )

    with open(result_path, "r", encoding="utf-8") as f:
        result = json.load(f)

    checkpoint_rows.append({
        "checkpoint": checkpoint,
        "macro_precision": result["macro_precision"],
        "macro_recall": result["macro_recall"],
        "macro_f1": result["macro_f1"],
    })

checkpoint_df = pd.DataFrame(checkpoint_rows)

base_rows = []

for model in MODELS:
    result_path = BASE_BASE_DIR / model / "test_results.json"

    with open(result_path, "r", encoding="utf-8") as f:
        result = json.load(f)

    base_rows.append({
        "model": model,
        "macro_precision": result["macro_precision"],
        "macro_recall": result["macro_recall"],
        "macro_f1": result["macro_f1"],
    })

base_df = pd.DataFrame(base_rows)

checkpoint_df.to_csv(
    OUT_DIR / "checkpoint_macro_metrics.csv",
    index=False,
)

base_df.to_csv(
    OUT_DIR / "base_model_macro_metrics.csv",
    index=False,
)

#print("Checkpoint metrics:")
#print(checkpoint_df)

#print("\nBase model metrics:")
#print(base_df)

sns.set_theme(style="whitegrid")

for metric, metric_title in METRICS.items():
    plt.figure(figsize=(10, 6))

    ax = sns.lineplot(
        data=checkpoint_df,
        x="checkpoint",
        y=metric,
        marker="o",
        linewidth=2.5,
        label="QwenGuard checkpoints",
    )

    for _, row in checkpoint_df.iterrows():
        ax.annotate(
            f"{row[metric]:.3f}",
            xy=(row["checkpoint"], row[metric]),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )

    for _, row in base_df.iterrows():
        ax.axhline(
            y=row[metric],
            linestyle="--",
            linewidth=2,
            label=f"{row['model']}: {row[metric]:.3f}",
        )

    ax.set_title(
        f"{metric_title} over checkpoints compared with base models"
    )
    ax.set_xlabel("Checkpoint")
    ax.set_ylabel(metric_title)
    ax.set_xticks(CHECKPOINTS)
    ax.set_ylim(0, 1)

    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    plt.tight_layout()

    output_path = (
        OUT_DIR
        / f"{metric}_checkpoints_vs_base_models.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(f"Saved: {output_path}")

In [ ]:
# Load per-PEGI metrics from our checkpoints

PEGIS = ["3", "7", "12", "16", "18"]

PER_PEGI_METRICS = {
    "Precision": "Precision",
    "Recall": "Recall",
    "F1": "F1",
    "F2": "F2",
}

per_pegi_rows = []

for checkpoint in CHECKPOINTS:
    result_path = (
        OUR_BASE_DIR
        / f"checkpoint_{checkpoint}"
        / "test_results.json"
    )

    with open(result_path, "r", encoding="utf-8") as f:
        result = json.load(f)

    for pegi in PEGIS:
        pegi_result = result["rating_per_label"][pegi]

        per_pegi_rows.append({
            "checkpoint": checkpoint,
            "PEGI": f"PEGI {pegi}",
            "Precision": pegi_result["Precision"],
            "Recall": pegi_result["Recall"],
            "F1": pegi_result["F1"],
            "F2": pegi_result["F2"],
        })

per_pegi_df = pd.DataFrame(per_pegi_rows)

per_pegi_df.to_csv(
    OUT_DIR / "checkpoint_per_pegi_metrics.csv",
    index=False,
)

#print("\nPer-PEGI checkpoint metrics:")
#print(per_pegi_df)


# Plot progress for each PEGI rating

pegi_palette = {
    "PEGI 3": "#488f31",
    "PEGI 7": "#efd50d",
    "PEGI 12": "#d1570b",
    "PEGI 16": "#B51022",
    "PEGI 18": "#6a3d9a",
}

for metric, metric_title in PER_PEGI_METRICS.items():
    plt.figure(figsize=(10, 6))

    ax = sns.lineplot(
        data=per_pegi_df,
        x="checkpoint",
        y=metric,
        hue="PEGI",
        hue_order=[f"PEGI {pegi}" for pegi in PEGIS],
        palette=pegi_palette,
        marker="o",
        linewidth=2.2,
    )

    ax.set_title(
        f"{metric_title} per PEGI rating over checkpoints"
    )
    ax.set_xlabel("Checkpoint")
    ax.set_ylabel(metric_title)
    ax.set_xticks(CHECKPOINTS)
    ax.set_ylim(0, 1)

    ax.legend(
        title="PEGI rating",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    plt.tight_layout()

    output_path = (
        OUT_DIR
        / f"{metric.lower().replace(' ', '_')}_per_pegi_over_checkpoints.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(f"Saved: {output_path}")